# Nível 1 — Parte A: dados e regras determinísticas

Nesta etapa, os cálculos são feitos exclusivamente com pandas. A LLM será usada apenas na Parte B.

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
DATA_PATH = Path('../dados/dados_nivel_1.json')
dados = json.loads(DATA_PATH.read_text(encoding='utf-8'))
TAXA_USD_BRL = dados['taxa_cambio_usd_brl']
df_bruto = pd.DataFrame(dados['operacoes'])
print(f'Taxa fixa usada: R$ {TAXA_USD_BRL:.2f} por USD')
print(f'Registros carregados: {len(df_bruto)}')

Taxa fixa usada: R$ 5.40 por USD
Registros carregados: 20


## 1. Diagnóstico da qualidade dos dados

Verificamos duplicidades, datas ausentes, moedas, valores inválidos e campos ausentes antes de aplicar as regras.

In [2]:
campos_esperados = {'id', 'cliente_id', 'data', 'valor', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao'}
ids_duplicados = df_bruto[df_bruto['id'].duplicated(keep=False)].sort_values('id')
datas_ausentes = df_bruto[df_bruto['data'].isna()]
campos_ausentes = sorted(campos_esperados - set(df_bruto.columns))
valores_invalidos = df_bruto[df_bruto['valor'].isna() | (df_bruto['valor'] <= 0)]
print('IDs duplicados:')
print(ids_duplicados[['id', 'cliente_id', 'valor', 'contraparte']].to_string(index=False))
print(f'\nDatas ausentes: {datas_ausentes["id"].tolist()}')
print(f'Moedas encontradas: {df_bruto["moeda"].value_counts().to_dict()}')
print(f'Campos ausentes: {campos_ausentes}')
print(f'Valores nulos ou não positivos: {len(valores_invalidos)}')

IDs duplicados:
     id cliente_id  valor         contraparte
OP-0007    CLI-A-3  17200 Epsilon Consultoria
OP-0007    CLI-A-3  17200 Epsilon Consultoria

Datas ausentes: ['OP-0017']
Moedas encontradas: {'BRL': 19, 'USD': 1}
Campos ausentes: []
Valores nulos ou não positivos: 0


### Decisões de limpeza

A análise do esquema indicou que `id` é a chave única de cada registro. Portanto, a filtragem de duplicidade é feita exclusivamente por `id`: mantenho a primeira ocorrência e descarto ocorrências posteriores com a mesma chave, independentemente dos demais campos. Isso evita contar duas vezes uma mesma operação. Em dados reais, registros com o mesmo `id` e campos divergentes deveriam ser enviados para uma fila de inconsistências para investigação.
- `OP-0007` aparece duas vezes e a segunda ocorrência é removida pela chave `id`.
- `OP-0017` não tem data, mas contém valor e observação explicando a falha do sistema; preservo o registro. Ele não participa da Regra 1, que depende de data.
- A operação em USD é preservada e convertida para BRL pela taxa fixa fornecida.
- Não há campos ausentes nem valores nulos ou não positivos que exijam descarte adicional.

In [3]:
df = df_bruto.copy()
for coluna in ['id', 'cliente_id', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao']:
    df[coluna] = df[coluna].fillna('').astype(str).str.strip() #Limpeza de strings: remoção de espaços em branco e conversão para string
df = df.drop_duplicates(subset='id', keep='first').reset_index(drop=True)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
df['valor_brl'] = df['valor'] * df['moeda'].map({'BRL': 1.0, 'USD': TAXA_USD_BRL})
df['data_ausente'] = df['data'].isna()
print(f'Registros após deduplicação: {len(df)}')
print(f'Registros preservados sem data: {int(df["data_ausente"].sum())}')
df.head()

Registros após deduplicação: 19
Registros preservados sem data: 1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,valor_brl,data_ausente
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,"18,100.00",False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,"17,300.00",False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,"18,800.00",False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,"3,300.00",False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,"25,900.00",False


## 2. Agregações descritivas

As duas agregações pedidas são calculadas no DataFrame limpo e usam `valor_brl`.

In [4]:
volume_por_cliente = (df.groupby('cliente_id', as_index=False)['valor_brl'].sum().rename(columns={'valor_brl': 'volume_total_brl'}).sort_values('volume_total_brl', ascending=False))
quantidade_por_canal = (df.groupby('canal', as_index=False)['id'].count().rename(columns={'id': 'quantidade_operacoes'}).sort_values('quantidade_operacoes', ascending=False))
print('Volume total por cliente:')
display(volume_por_cliente)
print('Quantidade de operações por canal:')
display(quantidade_por_canal)

Volume total por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,"79,500.00"
0,CLI-A-1,"57,500.00"
1,CLI-A-2,"52,900.00"
2,CLI-A-3,"48,500.00"
4,CLI-A-5,"16,900.00"
5,CLI-A-6,"10,200.00"


Quantidade de operações por canal:


,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## 3. Regra 1 — fracionamento

Sinalizamos todas as operações de um cliente em uma mesma data quando o grupo tem pelo menos três operações, soma superior a R$ 50.000,00 e nenhuma operação individual atinge R$ 20.000,00. Grupos sem data são excluídos porque não é possível estabelecer a mesma data com segurança.

In [5]:
grupos_por_data = (df.dropna(subset=['data']).groupby(['cliente_id', 'data'], as_index=True).agg(quantidade_operacoes=('id', 'size'), soma_brl=('valor_brl', 'sum'), max_operacao_brl=('valor_brl', 'max')))
grupos_fracionamento = grupos_por_data[(grupos_por_data['quantidade_operacoes'] >= 3) & (grupos_por_data['soma_brl'] > 50000) & (grupos_por_data['max_operacao_brl'] < 20000)]
chaves = pd.MultiIndex.from_frame(df[['cliente_id', 'data']])
df['regra_1_fracionamento'] = chaves.isin(grupos_fracionamento.index)
print('Grupos que atendem à Regra 1:')
display(grupos_fracionamento.reset_index())
print('Operações sinalizadas:')
display(df.loc[df['regra_1_fracionamento'], ['id', 'cliente_id', 'data', 'valor_brl']])

Grupos que atendem à Regra 1:


,cliente_id,data,quantidade_operacoes,soma_brl,max_operacao_brl
0,CLI-A-1,2026-03-09,3,"54,200.00","18,800.00"


Operações sinalizadas:


,id,cliente_id,data,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00"
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00"
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00"


## 4. Regra 2 — valor atípico

Calculamos a mediana por cliente e sinalizamos operações acima de cinco vezes essa mediana. A regra só é aplicada a clientes com pelo menos quatro operações. Todos esses cálculos são feitos em pandas.

In [6]:
df['quantidade__transacoes_cliente'] = df.groupby('cliente_id')['id'].transform('size')
df['mediana_cliente_brl'] = df.groupby('cliente_id')['valor_brl'].transform('median')
df['limite_atipico_brl'] = 5 * df['mediana_cliente_brl']
df['regra_2_valor_atipico'] = ((df['quantidade__transacoes_cliente'] >= 4) & (df['valor_brl'] > df['limite_atipico_brl']))
display(df.loc[df['regra_2_valor_atipico'], ['id', 'cliente_id', 'valor_brl', 'mediana_cliente_brl', 'limite_atipico_brl']])

,id,cliente_id,valor_brl,mediana_cliente_brl,limite_atipico_brl
12,OP-0013,CLI-A-4,"64,800.00","5,450.00","27,250.00"


## 5. Validação das regras

A Regra 1 deve capturar `CLI-A-1` em 2026-03-09: três operações somam R$ 54.200,00 e todas são menores que R$ 20.000,00. `CLI-A-3` na mesma data é parecido, mas soma R$ 48.500,00 e não deve ser sinalizado.

In [7]:
validacao_regra_1 = pd.DataFrame([
    {'caso': 'esperado positivo', 'cliente_id': 'CLI-A-1', 'data': '2026-03-09', 'soma_esperada_brl': 54200.00, 'sinalizado': bool(df.loc[(df['cliente_id'] == 'CLI-A-1') & (df['data'] == '2026-03-09'), 'regra_1_fracionamento'].any())},
    {'caso': 'parecido, mas abaixo do limite', 'cliente_id': 'CLI-A-3', 'data': '2026-03-05', 'soma_esperada_brl': 48500.00, 'sinalizado': bool(df.loc[(df['cliente_id'] == 'CLI-A-3') & (df['data'] == '2026-03-05'), 'regra_1_fracionamento'].any())},
 ])
display(validacao_regra_1)
assert bool(validacao_regra_1.loc[0, 'sinalizado']) is True
assert bool(validacao_regra_1.loc[1, 'sinalizado']) is False
print('Validação da Regra 1 concluída com sucesso.')

,caso,cliente_id,data,soma_esperada_brl,sinalizado
0,esperado positivo,CLI-A-1,2026-03-09,"54,200.00",True
1,"parecido, mas abaixo do limite",CLI-A-3,2026-03-05,"48,500.00",False


Validação da Regra 1 concluída com sucesso.


In [8]:
resumo_sinalizacoes = df[['id', 'cliente_id', 'data', 'valor_brl', 'regra_1_fracionamento', 'regra_2_valor_atipico']].copy()
resumo_sinalizacoes['quantidade_regras'] = resumo_sinalizacoes[['regra_1_fracionamento', 'regra_2_valor_atipico']].sum(axis=1)
display(resumo_sinalizacoes[resumo_sinalizacoes['quantidade_regras'] > 0])

,id,cliente_id,data,valor_brl,regra_1_fracionamento,regra_2_valor_atipico,quantidade_regras
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00",True,False,1
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00",True,False,1
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00",True,False,1
12,OP-0013,CLI-A-4,2026-03-24,"64,800.00",False,True,1


# Parte B — análise com LLM

A LLM recebe fatos calculados pelo pandas e produz somente interpretação e redação. A chave é carregada do `.env` e nunca é exibida ou salva no notebook.

In [9]:
import os
import time
import requests
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError
from typing import Literal

load_dotenv('../.env')
CLIENTE_ESCOLHIDO = 'CLI-A-1'
operacoes_cliente = df[df['cliente_id'] == CLIENTE_ESCOLHIDO].copy()
sinalizacoes_cliente = operacoes_cliente[['regra_1_fracionamento', 'regra_2_valor_atipico']].any()
fatos_cliente = {
    'cliente_id': CLIENTE_ESCOLHIDO,
    'quantidade_operacoes': int(len(operacoes_cliente)),
    'volume_total_brl': round(float(operacoes_cliente['valor_brl'].sum()), 2),
    'mediana_valor_brl': round(float(operacoes_cliente['valor_brl'].median()), 2),
    'operacoes': operacoes_cliente[['id', 'data', 'valor_brl', 'moeda', 'canal', 'tipo', 'contraparte']].assign(data=lambda tabela: tabela['data'].dt.strftime('%Y-%m-%d')).to_dict(orient='records'),
    'regras_sinalizadas': [nome for nome, acionada in {'fracionamento': bool(sinalizacoes_cliente['regra_1_fracionamento']), 'valor_atipico': bool(sinalizacoes_cliente['regra_2_valor_atipico'])}.items() if acionada]
}
fatos_cliente

{'cliente_id': 'CLI-A-1',
 'quantidade_operacoes': 4,
 'volume_total_brl': 57500.0,
 'mediana_valor_brl': 17700.0,
 'operacoes': [{'id': 'OP-0001',
   'data': '2026-03-09',
   'valor_brl': 18100.0,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0002',
   'data': '2026-03-09',
   'valor_brl': 17300.0,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0003',
   'data': '2026-03-09',
   'valor_brl': 18800.0,
   'moeda': 'BRL',
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Beta Servicos ME'},
  {'id': 'OP-0004',
   'data': '2026-03-21',
   'valor_brl': 3300.0,
   'moeda': 'BRL',
   'canal': 'boleto',
   'tipo': 'pagamento',
   'contraparte': 'Gama Distribuidora'}],
 'regras_sinalizadas': ['fracionamento']}

In [10]:
class Parecer(BaseModel):
    nivel_risco: Literal['baixo', 'médio', 'alto']
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

SCHEMA_PARECER = Parecer.model_json_schema()
INSTRUCAO_SAIDA = f'''Retorne SOMENTE um objeto JSON válido, sem markdown, sem comentários e sem texto antes ou depois. O objeto deve obedecer exatamente a este schema JSON: {json.dumps(SCHEMA_PARECER, ensure_ascii=False)}'''

def validar_parecer(resposta):
    try:
        payload = resposta if isinstance(resposta, dict) else json.loads(resposta)
        return {'valido': True, 'parecer': Parecer.model_validate(payload).model_dump(), 'erro': None}
    except (json.JSONDecodeError, ValidationError, TypeError) as erro:
        return {'valido': False, 'parecer': None, 'erro': f'{type(erro).__name__}: {erro}'}

PROMPT_V1 = f'''Analise o cliente abaixo sob a perspectiva de prevenção à lavagem de dinheiro.
Use somente os fatos fornecidos, explique os sinais de alerta. {INSTRUCAO_SAIDA}

Fatos: {json.dumps(fatos_cliente, ensure_ascii=False, default=str)}'''

PROMPT_V2 = f'''Você é um analista de triagem de PLD. Interprete os fatos calculados pelo sistema, sem refazer cálculos e sem inventar dados.
A regra determinística já sinalizou: {fatos_cliente['regras_sinalizadas']}. Explique se os fatos são compatíveis com essa tipologia, diferencie indício de prova e indique limitações.
{INSTRUCAO_SAIDA}

Fatos estruturados: {json.dumps(fatos_cliente, ensure_ascii=False, default=str)}'''

In [11]:
def _chamar_gemini_request(prompt):
    api_key = os.getenv('GEMINI_API_KEY')
    model = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash-lite')
    if not api_key:
        raise RuntimeError('GEMINI_API_KEY não configurada no arquivo .env')
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent'
    body = {
        'contents': [{'parts': [{'text': prompt}]}],
        'generationConfig': {
            'temperature': 0.2,
            'responseMimeType': 'application/json',
            'responseSchema': Parecer.model_json_schema()
        }
    }
    inicio = time.perf_counter()
    resposta_http = requests.post(url, headers={'x-goog-api-key': api_key, 'Content-Type': 'application/json'}, json=body, timeout=60)
    latencia_ms = round((time.perf_counter() - inicio) * 1000, 2)
    resposta_http.raise_for_status()
    resposta = resposta_http.json()
    texto = resposta['candidates'][0]['content']['parts'][0]['text']
    uso = resposta.get('usageMetadata', {})
    resultado = validar_parecer(texto)
    resultado.update({'modelo': model, 'latencia_ms': latencia_ms, 'tokens_entrada': uso.get('promptTokenCount'), 'tokens_saida': uso.get('candidatesTokenCount'), 'tokens_total': uso.get('totalTokenCount')})
    return resultado

_chamar_gemini_sem_retry = _chamar_gemini_request

def chamar_gemini(prompt, max_tentativas=3):
    status_retentativa = {429, 500, 502, 503, 504}
    inicio_total = time.perf_counter()
    ultimo_erro = None
    for tentativa in range(1, max_tentativas + 1):
        try:
            resultado = _chamar_gemini_sem_retry(prompt)
            if resultado.get('valido'):
                resultado.update({'tentativas': tentativa, 'latencia_total_ms': round((time.perf_counter() - inicio_total) * 1000, 2)})
                return resultado
            ultimo_erro = resultado.get('erro', 'resposta malformada')
        except requests.RequestException as erro:
            status = getattr(getattr(erro, 'response', None), 'status_code', None)
            ultimo_erro = f'{type(erro).__name__}: erro de comunicação com a API'
            if status not in status_retentativa:
                break
        except (ValueError, KeyError, IndexError, TypeError) as erro:
            ultimo_erro = f'{type(erro).__name__}: resposta inesperada da API'
        if tentativa < max_tentativas:
            time.sleep(2 ** (tentativa - 1))
    return {'valido': False, 'parecer': None, 'erro': ultimo_erro, 'modelo': os.getenv('GEMINI_MODEL', 'gemini-3.6-flash'), 'tentativas': tentativa, 'latencia_total_ms': round((time.perf_counter() - inicio_total) * 1000, 2), 'tokens_entrada': None, 'tokens_saida': None, 'tokens_total': None}

resultado_v1 = chamar_gemini(PROMPT_V1)
resultado_v2 = chamar_gemini(PROMPT_V2)
print('Prompt V1:', resultado_v1)
print('Prompt V2:', resultado_v2)

Prompt V1: {'valido': True, 'parecer': {'nivel_risco': 'médio', 'tipologia_suspeita': 'Fracionamento (Structuring / Smurfing)', 'red_flags': ['Múltiplas transferências enviadas no mesmo dia (2026-03-09) somando R$ 54.200,00', 'Duas operações de PIX para a mesma contraparte (Alfa Comercio LTDA) no mesmo dia em valores fracionados (R$ 18.100,00 e R$ 17.300,00)', 'Regra do sistema sinalizada para fracionamento'], 'justificativa': 'O cliente realizou três transferências de valor expressivo no mesmo dia (09/03/2026), totalizando R$ 54.200,00, incluindo duas operações via PIX direcionadas à Alfa Comercio LTDA em valores fracionados. Esse comportamento levanta suspeita de fracionamento intencional de operações para burlar limites operacionais ou mecanismos automatizados de monitoramento.'}, 'erro': None, 'modelo': 'gemini-3.6-flash', 'latencia_ms': 7750.53, 'tokens_entrada': 591, 'tokens_saida': 228, 'tokens_total': 1624, 'tentativas': 1, 'latencia_total_ms': 7751.58}
Prompt V2: {'valido': Tr

## Comparação dos prompts

O Prompt V1 é mais aberto e tende a deixar mais decisões para a LLM. O Prompt V2 delimita o papel do modelo, proíbe novos cálculos ou fatos inventados e impõe o schema JSON. A comparação deve considerar não apenas o risco retornado, mas também a qualidade das justificativas e a validade estrutural.